In [1]:
import pandas as pd
import sys, os
from pathlib import Path

import os
sys.path.append(str(Path(os.getcwd()).parent))
from utils.wrappers import measure_time_and_space, measure_time

In [2]:
@measure_time_and_space
def select_target_csvs(directory):
    # Define target filenames (use a set for O(1) lookup)
    target_files = {
        "MSFT.csv",
        "NVDA.csv",
        "AAPL.csv",
        "GOOGL.csv",
        "AMZN.csv",
        "META.csv",
        "TSLA.csv",
    }

    selected = []
    with os.scandir(directory) as entries:
        for entry in entries:
            if entry.is_file() and entry.name in target_files:
                selected.append(entry.path)
    return selected

@measure_time_and_space
def select_target_csvs_nonrecursive(directory):
    # Define target filenames (use a set for O(1) lookup)
    target_files = {
        "MSFT_data.csv",
        "NVDA_data.csv",
        "AAPL_data.csv",
        "GOOGL_data.csv",
        "AMZN_data.csv",
        "META_data.csv",
        "TSLA_data.csv",
    }

    selected_paths = []

    # Efficiently iterate through directory (not recursive)
    for filename in os.listdir(directory):
        if filename in target_files:
            selected_paths.append(os.path.join(directory, filename))

    return selected_paths

In [4]:
select_target_csvs(Path.cwd() / "csv")
select_target_csvs_nonrecursive(Path.cwd() / "csv")



[select_target_csvs] Time elapsed: 0.001241 seconds
[select_target_csvs] Peak memory: 3.52 KB
[select_target_csvs_nonrecursive] Time elapsed: 0.000465 seconds
[select_target_csvs_nonrecursive] Peak memory: 37.49 KB


[]

In [10]:
file_paths = select_target_csvs(Path.cwd() / "csv")

[select_target_csvs] Time elapsed: 0.001382 seconds
[select_target_csvs] Peak memory: 3.68 KB


In [34]:
@measure_time_and_space

def read_filtered_csvs(file_paths, chunksize=100000):
    """
    Reads multiple CSV files in chunks, filters rows with Date.year > 2024,
    and returns a single combined DataFrame.
    """
    combined_chunks = []  # list to store filtered chunks

    for path in file_paths:
        for chunk in pd.read_csv(path, parse_dates=['Date'], chunksize=chunksize, usecols=lambda col: col != 'Adj Close'):
            # filtered_chunk = chunk[chunk['Date'].dt.year > 2024]
            filtered_chunk = chunk
            if not filtered_chunk.empty:
                combined_chunks.append(filtered_chunk)

    # Concatenate all filtered chunks into a single DataFrame
    combined_df = pd.concat(combined_chunks, ignore_index=True) if combined_chunks else pd.DataFrame()
    return combined_df

combined_df = read_filtered_csvs(file_paths)
combined_df.to_csv("mag7_stocks.csv")

[read_filtered_csvs] Time elapsed: 0.117887 seconds
[read_filtered_csvs] Peak memory: 10508.13 KB


In [36]:
combined_df

,Date,Ticker,Open,High,Low,Close,Volume
0,1980-12-12,AAPL,0.128348,0.128906,0.128348,0.128348,469033600
1,1980-12-12,AAPL,0.128348,0.128906,0.128348,0.128348,469033600
2,1980-12-15,AAPL,0.122210,0.122210,0.121652,0.121652,175884800
3,1980-12-15,AAPL,0.122210,0.122210,0.121652,0.121652,175884800
4,1980-12-16,AAPL,0.113281,0.113281,0.112723,0.112723,105728000
...,...,...,...,...,...,...,...
95363,2025-09-29,TSLA,444.350006,450.980011,439.500000,443.209991,79491500
95364,2025-09-30,TSLA,441.519989,445.000000,433.119995,444.720001,74358000
95365,2025-10-01,TSLA,443.799988,462.290009,440.750000,459.459991,98122300
95366,2025-10-02,TSLA,470.540009,470.750000,435.570007,436.000000,137009000
